# Run reliability experiment on BKT

## Install pyBKT

In [ ]:
# Install the same pyBKT version used in the baseline

!git clone https://github.com/CAHLR/pyBKT.git
%cd pyBKT
!pip install .

fatal: destination path 'pyBKT' already exists and is not an empty directory.
/content/pyBKT
Processing /content/pyBKT
  Preparing metadata (setup.py) ... done
  Created wheel for pyBKT: filename=pyBKT-1.4.3-cp313-cp313-linux_x86_64.whl size=1130417 sha256=3362b75b0350d04b79c95790cadfe53aec41f3c2a5b707fb7cd759204d291e94
  Stored in directory: /tmp/pip-ephem-wheel-cache-k2lj88ll/wheels/23/39/9e/901a22284c9c3d36fb209c86485f4bb3c82d33b69d8e532f2c
Successfully built pyBKT
  Attempting uninstall: pyBKT
    Found existing installation: pyBKT 1.4.3
    Uninstalling pyBKT-1.4.3:
      Successfully uninstalled pyBKT-1.4.3


## Imports

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os
import time
import numpy as np
import pandas as pd


RELIABILITY_DIR = (
    "/content/drive/MyDrive/"
    "education-ml-research/ASSISTments2009/"
    "reliability_experiment/"
)

# Load the same datasets used for SAKT
train_df = pd.read_csv(RELIABILITY_DIR + "train.csv")
q1_test = pd.read_csv(RELIABILITY_DIR + "q1_test.csv")
q2_test = pd.read_csv(RELIABILITY_DIR + "q2_test.csv")
q3_test = pd.read_csv(RELIABILITY_DIR + "q3_test.csv")
q4_test = pd.read_csv(RELIABILITY_DIR + "q4_test.csv")

print("Training:", train_df.shape)
print("Q1:", q1_test.shape)
print("Q2:", q2_test.shape)
print("Q3:", q3_test.shape)
print("Q4:", q4_test.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_7444/1421340122.py:18: DtypeWarning: Columns (0: skill_name) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(RELIABILITY_DIR + "train.csv")


Training: (426580, 30)
Q1: (11444, 30)
Q2: (23524, 30)
Q3: (30184, 30)
Q4: (33802, 30)


/tmp/ipykernel_7444/1421340122.py:22: DtypeWarning: Columns (0: skill_name) have mixed types. Specify dtype option on import or set low_memory=False.
  q4_test = pd.read_csv(RELIABILITY_DIR + "q4_test.csv")


## Prepare the data for BKT

In [ ]:
def prepare_bkt_data(data):
    data = data[
        ["order_id", "user_id", "skill_id", "correct"]
    ].copy()

    data = data.dropna(subset=["skill_id"])

    data["skill_id"] = (
        data["skill_id"]
        .astype(int)
        .astype(str)
    )

    data["correct"] = pd.to_numeric(
        data["correct"],
        errors="coerce"
    )

    data = data[
        data["correct"].isin([0, 1])
    ].copy()

    data = data.sort_values(
        ["user_id", "order_id"]
    ).reset_index(drop=True)

    return data


bkt_train = prepare_bkt_data(train_df)
bkt_q1 = prepare_bkt_data(q1_test)
bkt_q2 = prepare_bkt_data(q2_test)
bkt_q3 = prepare_bkt_data(q3_test)
bkt_q4 = prepare_bkt_data(q4_test)

print("Training interactions:", len(bkt_train))
print("Q1 interactions:", len(bkt_q1))
print("Q2 interactions:", len(bkt_q2))
print("Q3 interactions:", len(bkt_q3))
print("Q4 interactions:", len(bkt_q4))

Training interactions: 372054
Q1 interactions: 9373
Q2 interactions: 19826
Q3 interactions: 25960
Q4 interactions: 31995


## Fit the model

In [ ]:
from pyBKT.models import Model

start = time.time()

bkt_model = Model(
    seed=42,
    num_fits=1,
    parallel=True
)

bkt_model.fit(
    data=bkt_train,
    defaults={
        "order_id": "order_id",
        "student_id": "user_id",
        "skill_name": "skill_id",
        "correct": "correct"
    }
)

print(f"BKT training time: {time.time() - start:.2f} seconds")
print("BKT fitting complete.")

BKT training time: 1.06 seconds
BKT fitting complete.


## Calculate AUCs

In [ ]:
auc_q1 = bkt_model.evaluate(
    data=bkt_q1,
    metric="auc"
)

auc_q2 = bkt_model.evaluate(
    data=bkt_q2,
    metric="auc"
)

auc_q3 = bkt_model.evaluate(
    data=bkt_q3,
    metric="auc"
)

auc_q4 = bkt_model.evaluate(
    data=bkt_q4,
    metric="auc"
)

print(f"Q1 AUC: {auc_q1:.6f}")
print(f"Q2 AUC: {auc_q2:.6f}")
print(f"Q3 AUC: {auc_q3:.6f}")
print(f"Q4 AUC: {auc_q4:.6f}")

Q1 AUC: 0.729531
Q2 AUC: 0.735898
Q3 AUC: 0.713820
Q4 AUC: 0.813358


## Save the data

In [ ]:
import pandas as pd
import os

# BKT reliability results
bkt_results = pd.DataFrame({
    "ability_quartile": ["Q1", "Q2", "Q3", "Q4"],
    "auc": [
        auc_q1,
        auc_q2,
        auc_q3,
        auc_q4
    ]
})

# Save to the reliability experiment folder
output_path = os.path.join(
    RELIABILITY_DIR,
    "bkt_reliability_aucs.csv"
)

bkt_results.to_csv(output_path, index=False)

print("Saved:", output_path)
print("\nBKT Reliability AUCs:")
print(bkt_results)

Saved: /content/drive/MyDrive/education-ml-research/ASSISTments2009/reliability_experiment/bkt_reliability_aucs.csv

BKT Reliability AUCs:
  ability_quartile     auc
0               Q1 0.72953
1               Q2 0.73590
2               Q3 0.71382
3               Q4 0.81336


## Results

The BKT model was trained once using the training dataset and then evaluated separately on four student ability quartiles.

The model achieved an AUC of 0.7295 for Q1, 0.7359 for Q2, 0.7138 for Q3, and 0.8134 for Q4. Thus, BKT performance varied substantially across student ability groups, with a difference of approximately 0.10 AUC between the lowest-performing group (Q3) and highest-performing group (Q4).

The most notable finding was the substantially higher performance for Q4 students, whose AUC of 0.8134 was considerably greater than the AUCs of the other three groups. However, performance did not increase consistently with ability, as Q3 had the lowest AUC despite representing a higher ability group than Q1 and Q2.

The overall BKT AUC from the original experiment was 0.7945. The quartile AUCs do not need to average to this value because AUC calculated across the entire test set is not equivalent to the average of AUCs calculated separately for subgroups.

Overall, these results provide preliminary evidence that student ability affects the predictive performance of BKT. The substantial variation in AUC across ability groups suggests that evaluating a knowledge tracing model using only an overall AUC may conceal differences in its predictive performance for different types of students.